# Auditoría de Datos: Pipeline de Riesgo Financiero SBS - BCRP
Este notebook documenta y consolida todos los pasos de extracción, inspección y validación matemática realizados durante la **Fase 1 (Auditoría de Fuentes)**.

Asegúrate de subir los archivos `.XLS` y `.xlsx` de la SBS y del BCRP al entorno (icono de carpeta a la izquierda) antes de ejecutar las celdas.


In [ ]:
import pandas as pd
import numpy as np
import os

# Verificar archivos en el entorno
archivos_actuales = os.listdir('.')
print("Archivos disponibles:", [f for f in archivos_actuales if f.endswith(('.XLS', '.xlsx'))])

Archivos disponibles: ['B-2362-en2024.XLS', 'B-2362-en2025.XLS', 'B-2362-en2023.XLS']


## 1. Auditoría de Morosidad (Cuadro B-2362)
Identifica la tasa de morosidad por tipo de crédito (Consumo e Hipotecario) para los 5 bancos del MVP y el promedio del sistema.

In [ ]:
sbs_mor_files = {
    '2023-01': 'B-2362-en2023.XLS',
    '2024-01': 'B-2362-en2024.XLS',
    '2025-01': 'B-2362-en2025.XLS'
}

bancos_target = {
    'BCP': ['crédito del perú', 'bcp'],
    'BBVA': ['bbva'],
    'Interbank': ['interbank'],
    'Scotiabank': ['scotiabank'],
    'Mibanco': ['mibanco'],
    'Total Banca Múltiple': ['total banca múltiple', 'total banca']
}

mor_data = []
for periodo, fname in sbs_mor_files.items():
    if not os.path.exists(fname): continue
    df = pd.read_excel(fname, sheet_name=0, header=None)
    header = df.iloc[5]

    row_cons, row_hipo = None, None
    for idx, val in enumerate(df[0]):
        if pd.notna(val):
            v_str = str(val).lower().strip()
            if 'créditos de consumo' in v_str: row_cons = idx
            elif 'hipotecarios para vivienda' in v_str: row_hipo = idx

    for std_name, patterns in bancos_target.items():
        found_col = None
        for c_idx, c_val in enumerate(header):
            if pd.notna(c_val) and any(p in str(c_val).lower() for p in patterns):
                found_col = c_idx
                break

        if found_col is not None:
            mor_data.append({'fecha': periodo, 'banco': std_name, 'tipo_credito': 'Consumo', 'indicador': 'Morosidad', 'valor': df.iloc[row_cons, found_col]})
            mor_data.append({'fecha': periodo, 'banco': std_name, 'tipo_credito': 'Hipotecario', 'indicador': 'Morosidad', 'valor': df.iloc[row_hipo, found_col]})

df_morosidad = pd.DataFrame(mor_data)
display(df_morosidad.head())

,fecha,banco,tipo_credito,indicador,valor
0,2023-01,BCP,Consumo,Morosidad,2.588294
1,2023-01,BCP,Hipotecario,Morosidad,2.659380
2,2023-01,BBVA,Consumo,Morosidad,2.917017
3,2023-01,BBVA,Hipotecario,Morosidad,2.275152
4,2023-01,Interbank,Consumo,Morosidad,3.262398


## 2. Auditoría de Tasas Activas MN
Controla la deriva estructural de columnas (corrimiento por error de Bank of China) y extrae las tasas anuales para Consumo e Hipotecario.

In [ ]:
sbs_ti_files = {
    '2023-01': 'B_TIActivaTipoCreditoEmpresaMN105021_2023.xlsx',
    '2024-01': 'B_TIActivaTipoCreditoEmpresaMN104956_2024.xlsx',
    '2025-01': 'B_TIActivaTipoCreditoEmpresaMN104751.xlsx'
}
ti_data = []

for periodo, fname in sbs_ti_files.items():
    if not os.path.exists(fname): continue
    df = pd.read_excel(fname, sheet_name='Reporte', header=None)
    header = df.iloc[6]

    banks_ti_idx = {'BCP': 4, 'BBVA': 2, 'Interbank': 9, 'Scotiabank': 7, 'Mibanco': 10}

    prom_col = None
    for c_idx, val in enumerate(header):
        if pd.notna(val) and 'promedio' in str(val).lower():
            prom_col = c_idx
            break
    if prom_col is not None:
        banks_ti_idx['Total Banca Múltiple'] = prom_col

    for std_name, col_idx in banks_ti_idx.items():
        ti_data.append({'fecha': periodo, 'banco': std_name, 'tipo_credito': 'Consumo', 'indicador': 'Tasa Activa MN', 'valor': df.iloc[44, col_idx]})
        ti_data.append({'fecha': periodo, 'banco': std_name, 'tipo_credito': 'Hipotecario', 'indicador': 'Tasa Activa MN', 'valor': df.iloc[51, col_idx]})

df_tasas = pd.DataFrame(ti_data)
display(df_tasas.head())

,fecha,banco,tipo_credito,indicador,valor
0,2023-01,BCP,Consumo,Tasa Activa MN,44.75
1,2023-01,BCP,Hipotecario,Tasa Activa MN,9.49
2,2023-01,BBVA,Consumo,Tasa Activa MN,39.23
3,2023-01,BBVA,Hipotecario,Tasa Activa MN,9.65
4,2023-01,Interbank,Consumo,Tasa Activa MN,56.25


## 3. Auditoría de Créditos Directos y Control Matemático
Extrae saldos contables y verifica que la morosidad oficial cruce matemáticamente (`Cartera Atrasada / Creditos Totales`).

In [ ]:
sbs_cd_files = {
    '2023-01': 'B-2334-en2023.XLS',
    '2024-01': 'B-2334-en2024.XLS',
    '2025-01': 'B-2334-en2025.XLS'
}
cd_data = []

for periodo, fname in sbs_cd_files.items():
    if not os.path.exists(fname): continue
    df = pd.read_excel(fname, sheet_name=0, header=None)

    for r_idx in range(len(df)):
        banco_val = str(df.iloc[r_idx, 0]).strip().lower()
        matched_banco = None
        for std_name, patterns in bancos_target.items():
            if any(p in banco_val for p in patterns):
                matched_banco = std_name
                break

        if matched_banco:
            cons_tot = df.iloc[r_idx, range(21, 27)].sum()
            cons_atr = df.iloc[r_idx, [23, 26]].sum()
            hipo_tot = df.iloc[r_idx, range(28, 31)].sum()
            hipo_atr = df.iloc[r_idx, 30]

            cd_data.extend([
                {'fecha': periodo, 'banco': matched_banco, 'tipo_credito': 'Consumo', 'indicador': 'Créditos Directos Totales', 'valor': cons_tot},
                {'fecha': periodo, 'banco': matched_banco, 'tipo_credito': 'Consumo', 'indicador': 'Cartera Atrasada', 'valor': cons_atr},
                {'fecha': periodo, 'banco': matched_banco, 'tipo_credito': 'Hipotecario', 'indicador': 'Créditos Directos Totales', 'valor': hipo_tot},
                {'fecha': periodo, 'banco': matched_banco, 'tipo_credito': 'Hipotecario', 'indicador': 'Cartera Atrasada', 'valor': hipo_atr}
            ])

df_creditos = pd.DataFrame(cd_data)

# Control Matemático
df_cd_pivot = df_creditos.pivot(index=['fecha', 'banco', 'tipo_credito'], columns='indicador', values='valor').reset_index()
df_cd_pivot['Morosidad Calculada (%)'] = (df_cd_pivot['Cartera Atrasada'] / df_cd_pivot['Créditos Directos Totales']) * 100

control_dq = pd.merge(df_cd_pivot, df_morosidad.rename(columns={'valor': 'Morosidad Publicada SBS (%)'}), on=['fecha', 'banco', 'tipo_credito'])
control_dq['Diferencia Absoluta'] = abs(control_dq['Morosidad Calculada (%)'] - control_dq['Morosidad Publicada SBS (%)'])
control_dq['Resultado DQ'] = np.where(control_dq['Diferencia Absoluta'] < 1e-4, '✅ Consistente', '❌ Diferencia Detectada')

display(control_dq[['fecha', 'banco', 'tipo_credito', 'Morosidad Calculada (%)', 'Morosidad Publicada SBS (%)', 'Resultado DQ']].head())

,fecha,banco,tipo_credito,Morosidad Calculada (%),Morosidad Publicada SBS (%),Resultado DQ
0,2023-01,BBVA,Consumo,2.917017,2.917017,✅ Consistente
1,2023-01,BBVA,Hipotecario,2.275152,2.275152,✅ Consistente
2,2023-01,BCP,Consumo,2.588294,2.588294,✅ Consistente
3,2023-01,BCP,Hipotecario,2.659380,2.659380,✅ Consistente
4,2023-01,Interbank,Consumo,3.262398,3.262398,✅ Consistente


## 4. Auditoría BCRP (Variables Macroeconómicas) y Tabla de Hechos Consolidada
Lectura de datos desde Excel (o API) usando mapeo de fechas de formato `Ene23` a `2023-01` y armado final del dataset.

In [ ]:
bcrp_mapping = {
    'Mensuales-20260908-231230.xlsx': ('Tasa de Referencia', 'PD04722MM', '%'),
    'Mensuales-20260908-231700.xlsx': ('Tipo de Cambio PEN/USD', 'PN01207PM', 'PEN/USD'),
    'Mensuales-20260908-232503.xlsx': ('Inflacion IPC (var% 12m)', 'PN01273PM', '%')
}
target_periods = {'Ene23': '2023-01', 'Ene24': '2024-01', 'Ene25': '2025-01'}
bcrp_data = []

for fname, (ind_name, code, unit) in bcrp_mapping.items():
    if not os.path.exists(fname): continue
    df_m = pd.read_excel(fname, sheet_name='Mensuales')
    for idx, row in df_m.iterrows():
        p_str = str(row.iloc[0]).strip()
        if p_str in target_periods:
            bcrp_data.append({
                'fecha': target_periods[p_str], 'banco': 'Macro',
                'tipo_credito': 'Macro', 'indicador': ind_name, 'valor': row.iloc[1]
            })

df_bcrp = pd.DataFrame(bcrp_data)

# Consolidación final eliminando variables auxiliares
df_creditos_final = df_creditos[df_creditos['indicador'] == 'Créditos Directos Totales']
df_fact_master = pd.concat([df_morosidad, df_tasas, df_creditos_final, df_bcrp], ignore_index=True)
df_fact_master['valor'] = df_fact_master['valor'].astype(float).round(4)

print(f"Tabla de Hechos Generada: {df_fact_master.shape[0]} registros en total.")
display(df_fact_master.sample(10))

# Exportar a CSV
df_fact_master.to_csv("fact_indicador_financiero.csv", index=False)

Tabla de Hechos Generada: 117 registros en total.


,fecha,banco,tipo_credito,indicador,valor
78,2023-01,Interbank,Consumo,Créditos Directos Totales,1.521643e+07
27,2025-01,BBVA,Hipotecario,Morosidad,2.077400e+00
89,2024-01,Scotiabank,Hipotecario,Créditos Directos Totales,1.133869e+07
75,2023-01,BCP,Hipotecario,Créditos Directos Totales,2.011036e+07
108,2023-01,Macro,Macro,Tasa de Referencia,7.750000e+00
72,2023-01,BBVA,Consumo,Créditos Directos Totales,1.041465e+07
13,2024-01,BCP,Hipotecario,Morosidad,2.729400e+00
107,2025-01,Total Banca Múltiple,Hipotecario,Créditos Directos Totales,6.693667e+07
97,2025-01,BBVA,Hipotecario,Créditos Directos Totales,1.726717e+07
34,2025-01,Total Banca Múltiple,Consumo,Morosidad,3.180500e+00
